In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from RRAM import *
from tqdm import tqdm

import pandas as pd

# Creo el excel donde voy a sacar todos los datos
df = pd.DataFrame(columns=['Tiempo simulacion', 'Voltaje', 'Campo Eléctrico', 'Corriente', 'Temperatura',
                  'Probabilidad de Generación', 'Probabilidad de Recombinación', 'Percolacion'])

In [ ]:
espesor_dispositivo = 10        # nm
Atom_size = 0.5                 # nm

eje_x = round(espesor_dispositivo / Atom_size)
eje_y = round(espesor_dispositivo / Atom_size)

num_trampas = 10

# FIXME: Hay una zona donde nunca se ponen trampas
actual_state = Generation.initial_state(eje_x, eje_y, num_trampas)


total_simulation_time = 1
num_pasos = 1000
paso_temporal = total_simulation_time / num_pasos

voltaje_final = 3

paso_guardar = 10

configuraciones_matriz = np.zeros((int((num_pasos / paso_guardar)), eje_x, eje_y))

# Configuraciones iniciales:
Temperatura = 350
Campo_Electrico = 0
voltaje = 0
simulation_time = 0
Corriente = 0

In [ ]:
for k in tqdm(range(0, num_pasos)):
    # Guardo el estado anterior
    last_state = actual_state

    simulation_time = paso_temporal*k

    # Calculo la corriente
    voltaje += voltaje_final * paso_temporal

    # Obtengo la corrriente, antes decido cual usar comprobando si ha percolado o no
    # TODO: Revisar por qué me dice que ha percolado si no lo ha hecho
    if Percolation.is_path(actual_state):
        # Si ha percolado uso la corriente de percolación
        # Corriente = CurentSolver.OmhCurrent(Temperatura, Campo_Electrico)
        print("Ha percolado")
    else:
        # Si no ha percolado uso la corriente de campo
        # TODO: REVISAR QUE LA CORRIENTE TIENE LAS UNIDADES CORRECTAS PORQUE NO CUADRAN VALORES.
        Corriente = 10000*CurentSolver.poole_frenkel(Temperatura, Campo_Electrico)

    # Obtengo los valores del campo eléctrico y la temperatura
    Campo_Electrico = SimpleElectricField(voltaje, espesor_dispositivo*1e-9)
    Temperatura = Temperature_Joule(voltaje, Corriente, T_0=350)

    # Calculo la probabilidad de generación o recombinación para ello recorro toda la matriz
    for i in range(eje_x):
        for j in range(eje_y):
            if actual_state[i, j] == 0:
                # TODO: REVISAR PROBABILIDAD QUE A VECES SALE MAYOR DE 1
                # TODO: HACER UN REESCALADO DE LOS VALORES PARA EVITAR TENER QUE TRABAJAR CON NUMEROS TAN GRANDES
                prob_generacion = Generation.generation(paso_temporal, Campo_Electrico, Temperatura)
                if (i == 0 and j == 0):
                    # genero un número aleatorio entre 0 y 1
                    random_number = np.random.rand()
                if random_number < prob_generacion:
                    actual_state[i, j] = 1  # Generación
            else:
                # TODO: REVISAR PROBABILIDAD QUE A VECES SALE MAYOR DE 1
                # TODO: HACER UN REESCALADO DE LOS VALORES PARA EVITAR TENER QUE TRABAJAR CON NUMEROS TAN GRANDES
                prob_recombinacion = Recombination.recombination(paso_temporal, i, Campo_Electrico, Temperatura)

                # genero un número aleatorio entre 0 y 1
                random_number = np.random.rand()
                if random_number < prob_recombinacion:
                    actual_state[i, j] = 0  # Recombinación

    # Guardo los datos en el excel
    df.loc[k] = [simulation_time, voltaje, Campo_Electrico, Corriente, Temperatura,
                 prob_generacion, prob_recombinacion, Percolation.is_path(actual_state)]

    # Guardo el estado actual CADA paso_guardar PASOS MONTECARLO
    if k % paso_guardar == 0:
        configuraciones_matriz[int(k / paso_guardar) - 1] = actual_state

# Guardo los datos en un excel
df.to_excel('resultados.xlsx', index=False)